Multilabel CNN for contraction number and duration 

In [11]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

matplotlib.use('QtAgg') # for GUI 
mne.set_log_level("CRITICAL")

Define Model 

In [12]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    count_output = Dense(num_classes, activation='softmax', name='count_output')(x)
    duration_output = Dense(1, activation='linear', name='duration_output')(x)

    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [13]:
def CNN_model_regression(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 

    model.add(Dense(2, activation='linear'))  # Add the output layer with linear to do regression 

    return model 

Loading in Data

In [14]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

# dummy column of duration 
#features_all = features_all.assign(Duration_zygo = np.random.randint(0, 6, size=np.shape(features_all)[0]))
#features_all = features_all.assign(Duration_corr = np.random.randint(0, 6, size=np.shape(features_all)[0]))


In [29]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"].astype(int).to_numpy(),
                    features_all_temp["Duration_corr"].astype(int).to_numpy()])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [26]:
print(y[0])

[0 1]


In [55]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1
continuous = True 

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model_regression(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    if not continuous: # not treating as continuous 
        model.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mse'],metrics=['accuracy', 'mae'] ) #think about metric 
        model_history_kfold = model.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                              validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                              epochs=epoch_num)
        
        scores = model.evaluate(X_test,[y_test[:,0], y_test[:,1]])
        cvScores_dur.append(scores[3] * 100)
        cvScores_contr.append(scores[4] * 100)

    else:
        model.compile( optimizer='adam',loss='mse',metrics=['mae'])
        model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) 

        scores = model.evaluate(X_test,y_test) # SEE WHAT SCORES PRINTS 
        cvScores.append(scores[0] * 100)

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

if continuous:
    model_history = model.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])
else:
    model_history = model.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 27s 80ms/step - loss: 27.8550 - mae: 1.9252 - val_loss: 4.1583 - val_mae: 1.4554
Epoch 2/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 89ms/step - loss: 3.7920 - mae: 1.4294 - val_loss: 2.5700 - val_mae: 1.2965
Epoch 3/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 24s 82ms/step - loss: 3.2345 - mae: 1.3794 - val_loss: 2.3564 - val_mae: 1.2544
Epoch 4/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 24s 83ms/step - loss: 3.2398 - mae: 1.3632 - val_loss: 2.5045 - val_mae: 1.2746
Epoch 5/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 23s 81ms/step - loss: 2.9391 - mae: 1.3387 - val_loss: 2.4402 - val_mae: 1.2692
Epoch 6/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 24s 83ms/step - loss: 2.8919 - mae: 1.3268 - val_loss: 2.5481 - val_mae: 1.3207
Epoch 7/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 24s 83ms/step - loss: 2.9790 - mae: 1.3653 - val_loss: 2.5026 - val_mae: 1.3004
Epoch 8/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 22s 77ms/step - loss: 2.6707 - mae: 1.2914 -

IndexError: list index out of range

In [56]:
scores

[1.9773004055023193, 1.150286316871643]

In [ ]:
# save model 
with open("multilabel.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
# cross validation results 
avgScores_dur = np.mean(cvScores_dur)
stdScores_dur = np.std(cvScores_dur)

avgScores_contr= np.mean(cvScores_contr)
stdScores_contr = np.std(cvScores_contr)

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

In [ ]:
# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   
y_pred_train_dur = np.argmax(y_pred_train_dur, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
y_pred_test_dur = np.argmax(y_pred_test_dur, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full, y_pred_train_contraction)   
accuracy_training_dur = accuracy_score(y_train_full, y_pred_train_dur)   

accuracy_test_contraction = accuracy_score(y_test, y_pred_test_contraction)  
accuracy_test_dur= accuracy_score(y_test, y_pred_test_dur)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full, y_pred_train_contraction, average='weighted')  
f1_training_dur = f1_score(y_train_full, y_pred_train_dur, average='weighted')  

f1_test_contraction = f1_score(y_test, y_pred_test_contraction, average='weighted')  
f1_test_dur = f1_score(y_test, y_pred_test_dur, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)    
print('-----------------------------------------')
print("Training Accuracy :", accuracy_training_dur)
print("Test Accuracy :", accuracy_test_dur)
print("Training F1 Score :", f1_training_dur)
print("Test F1 Score :", f1_test_dur)